### Imports / Pull in dataset

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
Sleep = pd.read_csv('../data/sleep_health_dataset.csv')
Sleep.dtypes

person_id                        int64
age                              int64
gender                          object
occupation                      object
bmi                            float64
country                         object
sleep_duration_hrs             float64
sleep_quality_score            float64
rem_percentage                 float64
deep_sleep_percentage          float64
sleep_latency_mins               int64
wake_episodes_per_night          int64
caffeine_mg_before_bed           int64
alcohol_units_before_bed       float64
screen_time_before_bed_mins      int64
exercise_day                     int64
steps_that_day                   int64
nap_duration_mins                int64
stress_score                   float64
work_hours_that_day            float64
chronotype                      object
mental_health_condition         object
heart_rate_resting_bpm           int64
sleep_aid_used                   int64
shift_work                       int64
room_temperature_celsius 

### Feature Remapping

1. Drop person ID as it is not relevant to the data.
2. Because `sleep_disorder_risk` uses cateogircal values for ordered data, convert to numerical.

In [20]:
Sleep = Sleep.drop(columns=['person_id'])

risk_map = {
    'Healthy': 0,
    'Mild': 1,
    'Moderate': 2,
    'Severe': 3
}
Sleep['sleep_disorder_risk'] = Sleep['sleep_disorder_risk'].map(risk_map)

### Check for unhealthy correlations/data leakage

In [21]:
numeric_df = Sleep.select_dtypes(include=['number'])
corr_matrix = numeric_df.corr()
corr_matrix.style.background_gradient(cmap='coolwarm').format(precision=2)

,age,bmi,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,sleep_latency_mins,wake_episodes_per_night,caffeine_mg_before_bed,alcohol_units_before_bed,screen_time_before_bed_mins,exercise_day,steps_that_day,nap_duration_mins,stress_score,work_hours_that_day,heart_rate_resting_bpm,sleep_aid_used,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,cognitive_performance_score,sleep_disorder_risk,felt_rested
age,1.00,0.10,-0.00,-0.03,0.00,-0.47,0.00,0.11,0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,-0.07,0.06,-0.02
bmi,0.10,1.00,0.00,-0.06,0.00,-0.05,0.00,0.24,0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,-0.00,-0.01,0.00,0.01,0.00,-0.04,0.24,-0.05
sleep_duration_hrs,-0.00,0.00,1.00,0.65,0.06,0.05,-0.13,-0.11,-0.02,-0.00,-0.08,-0.00,0.00,-0.06,-0.50,-0.44,-0.05,0.11,-0.23,0.00,-0.00,0.62,-0.47,0.44
sleep_quality_score,-0.03,-0.06,0.65,1.00,0.25,0.09,-0.29,-0.38,-0.05,-0.08,-0.08,0.02,0.01,-0.02,-0.64,-0.37,-0.07,0.05,-0.26,-0.02,-0.01,0.86,-0.69,0.48
rem_percentage,0.00,0.00,0.06,0.25,1.00,0.12,-0.03,-0.07,-0.00,-0.37,0.01,0.17,0.09,0.00,-0.16,-0.06,-0.04,-0.01,-0.02,-0.00,0.00,0.45,-0.21,0.12
deep_sleep_percentage,-0.47,-0.05,0.05,0.09,0.12,1.00,-0.02,-0.08,-0.00,-0.12,0.01,0.29,0.16,0.00,-0.09,-0.04,-0.05,-0.01,-0.01,0.00,-0.00,0.28,-0.11,0.05
sleep_latency_mins,0.00,0.00,-0.13,-0.29,-0.03,-0.02,1.00,0.03,0.32,-0.00,0.29,-0.01,-0.00,-0.01,0.17,0.05,0.02,0.00,0.02,0.00,0.01,-0.23,0.33,-0.13
wake_episodes_per_night,0.11,0.24,-0.11,-0.38,-0.07,-0.08,0.03,1.00,0.00,0.13,0.00,-0.00,0.00,-0.01,0.18,0.08,0.02,-0.01,0.14,-0.00,-0.00,-0.29,0.35,-0.23
caffeine_mg_before_bed,0.00,0.00,-0.02,-0.05,-0.00,-0.00,0.32,0.00,1.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,-0.07,0.10,-0.03
alcohol_units_before_bed,-0.00,-0.00,-0.00,-0.08,-0.37,-0.12,-0.00,0.13,-0.00,1.00,-0.01,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,0.00,-0.01,-0.24,0.11,-0.04


### Remove data leakage

Sleep quality score and cognitive performance score are forms of data leakage/multicollinearality. Specifically, sleep_quality_score is subjective data collected from the patients. It is probably best if we do not allow the model to learn that "if someone said they slept good, they likely felt rested". Cognitive performance score is a result of calculated aggregate from other features, and is best left out to make the model less "shallow".

In [22]:
cols_to_drop = [
    'sleep_quality_score',
    'cognitive_performance_score'
]
Sleep = Sleep.drop(columns=cols_to_drop)